# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dipanshurdev/ML-Assignments/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# Setup
!pip install -q duckdb pandas
import duckdb
import pandas as pd

# Authenticate (Requires HF_TOKEN set in Colab Secrets or Env Var)
# import os
# hf_token = os.environ.get('HF_TOKEN', 'YOUR_TOKEN_HERE')
# duckdb.sql(f"INSTALL httpfs; LOAD httpfs; SET bearer_token='{hf_token}';")

# For this notebook we define the base path to query the HF dataset directly
BASE_URL = "hf://datasets/FlyRank/internship-warehouse/data/fact_content_daily_performance"
TARGET_MONTH = "2026-03"
QUERY_PATH = f"{BASE_URL}/month={TARGET_MONTH}/*"

## 1. Unit of analysis + time window

**Unit of analysis:** One row = one content item's search performance for a single day.
**Time window:** A single mid-panel month, `2026-03`.

In [ ]:
# Verify Grain: report_date, client_id, content_id should be unique
query_grain = f"""
SELECT report_date, client_id, content_id, COUNT(*) as c 
FROM '{QUERY_PATH}'
GROUP BY report_date, client_id, content_id 
HAVING c > 1 
LIMIT 5
"""
print("Grain Violations (Should be empty):")
display(duckdb.sql(query_grain).df())

## 2. Fields: feature / label / context / excluded

- **Feature:** `gsc_impressions`, `ga4_sessions`, `content_age_days`. These are knowable at the end of the day before predicting the daily CTR.
- **Label:** `ctr` (calculated as `gsc_clicks / gsc_impressions`). This is the target to predict.
- **Context:** `report_date`, `client_id`, `content_id`. Used only for joins and grouping.
- **Excluded:** `ga4_pageviews` and `ga4_scroll_events`. These represent on-page engagement that happens *after* the click, meaning they are future information relative to the search click decision.

In [ ]:
# Counts and Windows Check
query_counts = f"""
SELECT 
    COUNT(*) as total_rows,
    MIN(report_date) as start_date,
    MAX(report_date) as end_date
FROM '{QUERY_PATH}'
"""
print("Counts and Window Span:")
display(duckdb.sql(query_counts).df())

## 3. Verify it with queries (grain, counts, missing values, windows)

Queries testing grain, window, and data availability are structured here.

In [ ]:
# Availability Check (gsc_data_available is TRUE)
query_availability = f"""
SELECT 
    COUNT(*) as total_rows,
    COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) as gsc_available_rows
FROM '{QUERY_PATH}'
"""
print("GSC Data Availability:")
display(duckdb.sql(query_availability).df())

## Features + The Trap

Below we extract 5 valid features and intentionally introduce a leakage trap (`gsc_clicks`).

In [ ]:
# Building a 5-feature frame and the trap
query_features = f"""
SELECT
    -- 1. Context
    report_date,
    client_id,
    content_id,
    
    -- 2. Target Label
    TRY_CAST(gsc_clicks AS FLOAT) / NULLIF(gsc_impressions, 0) as ctr_label,
    
    -- 3. Features (Knowable before/at decision time)
    gsc_impressions,           -- Volume context available for CTR
    content_age_days,          -- Knowable since content creation
    ga4_sessions,              -- Historic traffic context
    gsc_avg_position,          -- Rank proxy (if we assume rank is known before click)
    word_count,                -- Static content feature
    
    -- 4. THE TRAP (Leakage)
    -- Including clicks directly as a feature when predicting CTR will perfectly leak the answer.
    gsc_clicks as TRAP_leakage_feature
    
FROM '{QUERY_PATH}'
WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
LIMIT 100
"""

df_features = duckdb.sql(query_features).df()
print("Features + Trap Frame (Showing first 5 rows):")
display(df_features.head())

# Demonstrating the leakage correlation:
correlation = df_features['TRAP_leakage_feature'].corr(df_features['ctr_label'] * df_features['gsc_impressions'])
print(f"Correlation between trap and (ctr * impressions): {correlation:.4f}")

# REMOVING THE TRAP
df_features = df_features.drop(columns=['TRAP_leakage_feature'])
print("\nTrap removed. Honest features remaining:", list(df_features.columns))

## 4. Data limits

- **What this data can never tell you:** The specific search query string that triggered the impression. The daily performance fact table is rolled up to the content level. To understand exactly *what* terms people searched, we would need to join `fact_content_query_90d`, being careful of time-window overlaps.
- Unbalanced History: The history depth differs by client, so a global month partition might include some clients with full history and others missing data entirely before their `ga4_data_start`.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.